In [1]:
import os

directory_path = '/Users/risky/project/personal/mba/thesis_v2/data/sideways_test'
file_names = os.listdir(directory_path)
print(file_names)

['CFCL.json', 'NIMB.json', 'GUFL.json', 'HRL.json', 'ULHC.json', 'SNLI.json', 'NABIL.json', 'RAWA.json', 'MATRI.json', 'NRIC.json']


In [2]:
import json

for file_name in file_names:
    file_path = os.path.join(directory_path, file_name)
    with open(file_path, 'r') as file:
        if(file_name == '.DS_Store' or file_name !="MATRI.json"):
            continue
        data = json.load(file)
        stock_name = file_name.split('.')[0]
        print(f"{file_name} -> {len(data)}")

MATRI.json -> 131


In [3]:
for entry in data:
    stock = entry['Symbol']
    date = entry['Date']
    open_price = entry['Open']
    high_price = entry['High']
    low_price = entry['Low']
    print(f"{stock} ({date}) -> Open: {open_price}, ⬆️ High: {high_price}, ⬇️ Low: {low_price}")

MATRI (2025-03-16) -> Open: 1273.1, ⬆️ High: 1322, ⬇️ Low: 1247
MATRI (2025-03-12) -> Open: 1308, ⬆️ High: 1333, ⬇️ Low: 1257
MATRI (2025-03-11) -> Open: 1270.1, ⬆️ High: 1294, ⬇️ Low: 1244.7
MATRI (2025-03-10) -> Open: 1293.6, ⬆️ High: 1320, ⬇️ Low: 1276.7
MATRI (2025-03-09) -> Open: 1328.9, ⬆️ High: 1329, ⬇️ Low: 1290
MATRI (2025-03-06) -> Open: 1325, ⬆️ High: 1326, ⬇️ Low: 1300.1
MATRI (2025-03-05) -> Open: 1324, ⬆️ High: 1410, ⬇️ Low: 1300
MATRI (2025-03-04) -> Open: 1340, ⬆️ High: 1340, ⬇️ Low: 1309
MATRI (2025-03-03) -> Open: 1333, ⬆️ High: 1362, ⬇️ Low: 1333
MATRI (2025-03-02) -> Open: 1350, ⬆️ High: 1365, ⬇️ Low: 1330
MATRI (2025-02-27) -> Open: 1374, ⬆️ High: 1374, ⬇️ Low: 1333
MATRI (2025-02-25) -> Open: 1332, ⬆️ High: 1352, ⬇️ Low: 1325
MATRI (2025-02-24) -> Open: 1360, ⬆️ High: 1370.1, ⬇️ Low: 1327.1
MATRI (2025-02-23) -> Open: 1340.2, ⬆️ High: 1371.9, ⬇️ Low: 1333
MATRI (2025-02-20) -> Open: 1360, ⬆️ High: 1410, ⬇️ Low: 1350
MATRI (2025-02-18) -> Open: 1407, ⬆️ High: 1407,

In [4]:
from termcolor import colored
from tabulate import tabulate

table = []
for entry in data:
    stock = entry['Symbol']
    date = entry['Date']
    open_price = entry['Open']
    high_price = entry['High']
    low_price = entry['Low']
    percent_change = entry['Percent Change']
    high_percent_change = round(((float(high_price) - float(open_price)) / float(open_price)) * 100, 2)
    low_percent_change = round(((float(low_price) - float(open_price)) / float(open_price)) * 100, 2)
    high_text = colored(f"⬆️ High: {high_price} ({high_percent_change})", "green")
    low_text = colored(f"⬇️ Low: {low_price} ({low_percent_change})", "red")
    table.append([stock, date, open_price, high_text, low_text])

print(tabulate(table, headers=["Stock", "Date", "Open", "High", "Low"], tablefmt="grid"))

+---------+------------+--------+------------------------+------------------------+
| Stock   | Date       |   Open | High                   | Low                    |
+=========+============+========+========================+========================+
| MATRI   | 2025-03-16 | 1273.1 | ⬆️ High: 1322 (3.84)   | ⬇️ Low: 1247 (-2.05)   |
+---------+------------+--------+------------------------+------------------------+
| MATRI   | 2025-03-12 | 1308   | ⬆️ High: 1333 (1.91)   | ⬇️ Low: 1257 (-3.9)    |
+---------+------------+--------+------------------------+------------------------+
| MATRI   | 2025-03-11 | 1270.1 | ⬆️ High: 1294 (1.88)   | ⬇️ Low: 1244.7 (-2.0)  |
+---------+------------+--------+------------------------+------------------------+
| MATRI   | 2025-03-10 | 1293.6 | ⬆️ High: 1320 (2.04)   | ⬇️ Low: 1276.7 (-1.31) |
+---------+------------+--------+------------------------+------------------------+
| MATRI   | 2025-03-09 | 1328.9 | ⬆️ High: 1329 (0.01)   | ⬇️ Low: 1290 (-2.

In [5]:
from termcolor import colored
from tabulate import tabulate


def highlight_changes(data, high_threshold, low_threshold, daily_investment=10000):

    table = []
    stocks_on_hold = 0
    profit_booked =0
    for entry in data:
        stock = entry["Symbol"]
        date = entry["Date"]
        open_price = entry["Open"]
        high_price = entry["High"]
        low_price = entry["Low"]
        high_percent_change = round(
            ((float(high_price) - float(open_price)) / float(open_price)) * 100, 2
        )
        low_percent_change = round(
            ((float(low_price) - float(open_price)) / float(open_price)) * 100, 2
        )
        high_text = f"⬆️ High: {high_price} ({high_percent_change})"
        low_text = f"⬇️ Low: {low_price} ({low_percent_change})"
        
        buying_price = open_price*((100-low_threshold)/100)
        selling_price = open_price*((100+high_threshold)/100)
        
        buy_unit = daily_investment/buying_price
        sell_unit = buy_unit
        
        buy_possible = buying_price>low_price
        sell_possible = selling_price<high_price
        
        if buy_possible:
            stocks_on_hold += buy_unit
            profit_booked += daily_investment
        if sell_possible:
            stocks_on_hold -= sell_unit
            profit_booked -= daily_investment

        if high_percent_change > high_threshold:
            high_text += " ✅"
        if low_percent_change < low_threshold:
            low_text += " ✅"

        high_text = colored(high_text, "green")
        low_text = colored(low_text, "red")
        
        

        table.append([stock, date, open_price, high_text, low_text])

    print(
        tabulate(
            table, headers=["Stock", "Date", "Open", "High", "Low"], tablefmt="grid"
        )
    )


highlight_changes(data, high_threshold=1.5, low_threshold=-1.5)

TypeError: can't multiply sequence by non-int of type 'float'

In [14]:
def count_successful_changes(data, high_threshold, low_threshold):
    total_success_highs = 0
    total_success_lows = 0
    total_success_both = 0

    for entry in data:
        open_price = entry['Open']
        high_price = entry['High']
        low_price = entry['Low']
        high_percent_change = round(((float(high_price) - float(open_price)) / float(open_price)) * 100, 2)
        low_percent_change = round(((float(low_price) - float(open_price)) / float(open_price)) * 100, 2)

        high_success = high_percent_change > high_threshold
        low_success = low_percent_change < low_threshold

        if high_success:
            total_success_highs += 1
        if low_success:
            total_success_lows += 1
        if high_success and low_success:
            total_success_both += 1

    return total_success_highs, total_success_lows, total_success_both

In [12]:
import json
table = [["Stock Name", "Total Entries", "Highs", "Lows", "Both"]]

for file_name in file_names:
    file_path = os.path.join(directory_path, file_name)
    with open(file_path, "r") as file:
        if file_name == ".DS_Store":
            continue
        data = json.load(file)
        high_threshold = 0.5
        low_threshold = -0.5
        day_investment = 100000
        total_success_highs, total_success_lows, total_success_both = count_successful_changes(data, high_threshold, low_threshold)
        stock_name = file_name.split(".")[0]
        
        table.append([stock_name, len(data), total_success_highs, total_success_lows, total_success_both])
print(tabulate(table, headers="firstrow", tablefmt="grid"))

+--------------+-----------------+---------+--------+--------+
| Stock Name   |   Total Entries |   Highs |   Lows |   Both |
+==============+=================+=========+========+========+
| CFCL         |             279 |     219 |    232 |    172 |
+--------------+-----------------+---------+--------+--------+
| NIMB         |             279 |     180 |    216 |    117 |
+--------------+-----------------+---------+--------+--------+
| GUFL         |             279 |     210 |    215 |    146 |
+--------------+-----------------+---------+--------+--------+
| HRL          |             274 |     227 |    200 |    153 |
+--------------+-----------------+---------+--------+--------+
| ULHC         |             279 |     245 |    213 |    179 |
+--------------+-----------------+---------+--------+--------+
| SNLI         |             279 |     232 |    174 |    127 |
+--------------+-----------------+---------+--------+--------+
| NABIL        |             279 |     137 |    220 |  